In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import (
    balanced_accuracy_score, f1_score, roc_auc_score,
    precision_recall_curve, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay,
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import matplotlib.pyplot as plt
from feature_sets import FEATURE_SETS, TARGET, resolve_features  

DATA_PATH    = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/modelling_panel.parquet")
RESULTS_DIR  = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/results/tier1")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(DATA_PATH)
panel = panel.sort_values("time_id").reset_index(drop=True)
print(f"Panel shape: {panel.shape}  |  class balance:\n{panel[TARGET].value_counts()}")


Panel shape: (51877, 84)  |  class balance:
Disrupted
0    30114
1    21763
Name: count, dtype: int64


In [3]:
# Temporal split 70 / 10 / 20
n_times   = panel["time_id"].nunique()
train_cut = int(n_times * 0.70)
val_cut   = int(n_times * 0.80)

train_ids = panel["time_id"].unique()[:train_cut]
val_ids   = panel["time_id"].unique()[train_cut:val_cut]
test_ids  = panel["time_id"].unique()[val_cut:]

mask_train = panel["time_id"].isin(train_ids)
mask_val   = panel["time_id"].isin(val_ids)
mask_test  = panel["time_id"].isin(test_ids)


def evaluate(y_true, y_prob, threshold=0.5, name=""):
    y_pred = (y_prob >= threshold).astype(int)
    ba  = balanced_accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    print(f"  [{name}]  BA={ba:.4f}  F1={f1:.4f}  AUC={auc:.4f}  θ={threshold:.3f}")
    return {"Balanced Accuracy": ba, "F1-Score": f1, "AUC-ROC": auc, "Threshold": threshold}


def best_f1_threshold(y_true, y_prob):
    """Find decision threshold that maximises F1 on the supplied set."""
    prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
    f1s  = 2 * prec * rec / (prec + rec + 1e-8)
    idx  = np.argmax(f1s)
    return float(thresholds[min(idx, len(thresholds) - 1)])

def make_models():
    return {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        "XGBoost": None,  # built inside loop (needs scale_pos_weight)
    }



In [4]:
all_results = []

for fs_name, fs_cols in FEATURE_SETS.items():
    print(f"\n{'='*60}")
    print(f"Feature set: {fs_name}  ({len(fs_cols)} features requested)")

    feats = resolve_features(fs_cols, panel.columns)
    print(f"  Available : {len(feats)} features")

    X_train_raw = panel.loc[mask_train, feats].fillna(0)
    y_train     = panel.loc[mask_train, TARGET].astype(int)

    X_val_raw   = panel.loc[mask_val,   feats].fillna(0)
    y_val       = panel.loc[mask_val,   TARGET].astype(int)

    X_test_raw  = panel.loc[mask_test,  feats].fillna(0)
    y_test      = panel.loc[mask_test,  TARGET].astype(int)

    # SMOTE on training only
    sm = SMOTE(random_state=42)
    X_train_res, y_train_res = sm.fit_resample(X_train_raw, y_train)

    # Scale
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_res)
    X_val_sc   = scaler.transform(X_val_raw)
    X_test_sc  = scaler.transform(X_test_raw)

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
    }

    for model_name, model in models.items():
        print(f"\n  ── {model_name} ──")
        model.fit(X_train_sc, y_train_res)

        # Threshold from validation set
        val_probs = model.predict_proba(X_val_sc)[:, 1]
        opt_threshold = best_f1_threshold(y_val, val_probs)

        # Final evaluation on test set
        test_probs = model.predict_proba(X_test_sc)[:, 1]
        metrics    = evaluate(y_test, test_probs, threshold=opt_threshold,
                              name=f"{model_name} | {fs_name}")
        metrics.update({"Model": model_name, "Feature Set": fs_name,
                        "n_features": len(feats)})
        all_results.append(metrics)

        # Classification report
        y_pred_test = (test_probs >= opt_threshold).astype(int)
        print(classification_report(y_test, y_pred_test, target_names=["Not Disrupted", "Disrupted"]))

        # Feature importance (XGBoost only)
        if model_name == "XGBoost":
            if model_name == "XGBoost":
                imp = pd.Series(model.feature_importances_, index=feats)
                imp.to_csv(RESULTS_DIR / f"xgb_importance_{fs_name.replace('+','')}.csv")

                # ── Rename labels to be human-readable in the figure ─────────────────
                LABEL_MAP = {
                    # Topology
                    "topo_src_degree":          "Degree (source)",
                    "topo_tgt_degree":          "Degree (target)",
                    "topo_src_betweenness":     "Betweenness (source)",
                    "topo_tgt_betweenness":     "Betweenness (target)",
                    "topo_src_closeness":       "Closeness (source)",
                    "topo_tgt_closeness":       "Closeness (target)",
                    "topo_src_clustering":      "Clustering (source)",
                    "topo_tgt_clustering":      "Clustering (target)",
                    "topo_src_eigenvector":     "Eigenvector (source)",
                    "topo_tgt_eigenvector":     "Eigenvector (target)",
                    "topo_edge_betweenness":    "Edge betweenness",
                    "topo_common_neighbours":   "Common neighbours",
                    # Operational
                    "rides_planned":            "Rides planned",
                    "stop_count":               "Stop count",
                    "disrupted_lag1":           "Disrupted $t-1$",
                    "disrupted_lag2":           "Disrupted $t-2$",
                    "disrupted_lag3":           "Disrupted $t-3$",
                    "delay_freq_3m":            "Delay freq. (3-month)",
                    "delay_freq_6m":            "Delay freq. (6-month)",
                    # Weather
                    "DR":  "Precip. days (DR)",
                    "RH":  "Precipitation (RH)",
                    "SQ":  "Sunshine (SQ)",
                    "TG":  "Mean temp. (TG)",
                    "TN":  "Min. temp. (TN)",
                    "TX":  "Max. temp. (TX)",
                    "RHX": "Max. precip. (RHX)",
                    # SES
                    "source_SES_Score_Wealth_Avg":     "Wealth score (source)",
                    "source_SES_Score_Education_Avg":  "Education score (source)",
                    "source_TotalVandalism":           "Vandalism (source)",
                    "source_Remoteness_Index":         "Remoteness (source)",
                    "target_SES_Score_Wealth_Avg":     "Wealth score (target)",
                    "target_SES_Score_Education_Avg":  "Education score (target)",
                    "target_TotalVandalism":           "Vandalism (target)",
                    "target_Remoteness_Index":         "Remoteness (target)",
                }

                imp_top = imp.nlargest(20).sort_values(ascending=True)
                imp_top.index = [LABEL_MAP.get(f, f) for f in imp_top.index]

                # ── Figure ────────────────────────────────────────────────────────────
                n_bars = len(imp_top)
                fig_h  = max(4.5, n_bars * 0.32)          # scale height with bar count
                fig, ax = plt.subplots(figsize=(6.5, fig_h))

                bars = ax.barh(
                    imp_top.index,
                    imp_top.values,
                    edgecolor="white",
                    linewidth=0.4,
                    height=0.65,
                )

                # Value labels at the end of each bar
                for bar, val in zip(bars, imp_top.values):
                    ax.text(
                        val + imp_top.values.max() * 0.01,
                        bar.get_y() + bar.get_height() / 2,
                        f"{val:.3f}",
                        va="center",
                        ha="left",
                        fontsize=8,
                        color="#333333",
                    )

                # Axes formatting
                ax.set_xlabel("Feature importance (gain)", labelpad=6)
                ax.set_xlim(0, imp_top.values.max() * 1.18)   # room for value labels
                ax.set_title(
                    f"XGBoost feature importance — {fs_name}",
                    pad=10,
                    fontweight="bold",
                )

                # Subtle dividing lines between feature groups (only for T+O+W+S)
                if fs_name == "T+O+W+S":
                    # Identify approximate group boundaries in the sorted bar list
                    ses_labels   = {"Wealth score (source)", "Education score (source)",
                                    "Vandalism (source)",    "Remoteness (source)",
                                    "Wealth score (target)", "Education score (target)",
                                    "Vandalism (target)",    "Remoteness (target)"}
                    weather_labels = {"Precip. days (DR)", "Precipitation (RH)",
                                    "Sunshine (SQ)",     "Mean temp. (TG)",
                                    "Min. temp. (TN)",   "Max. temp. (TX)",
                                    "Max. precip. (RHX)"}
                    label_list = list(imp_top.index)
                    for i, lbl in enumerate(label_list[:-1]):
                        next_lbl = label_list[i + 1]
                        # Draw a faint line when crossing from one group to another
                        in_ses  = lbl in ses_labels
                        in_wea  = lbl in weather_labels
                        nxt_ses = next_lbl in ses_labels
                        nxt_wea = next_lbl in weather_labels
                        if (in_ses != nxt_ses) or (in_wea != nxt_wea and not (in_ses or nxt_ses)):
                            ax.axhline(
                                y=i + 0.5,
                                color="#aaaaaa",
                                linewidth=0.7,
                                linestyle=":",
                            )

                ax.spines["left"].set_visible(False)
                ax.tick_params(axis="y", length=0)
                fig.tight_layout()

                fname = RESULTS_DIR / f"xgb_importance_{fs_name.replace('+','')}.png"
                fig.savefig(fname, dpi=300)
                plt.close()
                print(f"    Saved: {fname}")

# Compile and save results table 
results_df = pd.DataFrame(all_results)
results_df = results_df[["Model", "Feature Set", "Balanced Accuracy",
                          "F1-Score", "AUC-ROC", "Threshold", "n_features"]]
results_df = results_df.sort_values(["Model", "Feature Set"])

print("\n" + "="*60)
print("TIER 1 — FULL RESULTS TABLE")
print("="*60)
print(results_df.round(4).to_string(index=False))

results_df.to_csv(RESULTS_DIR / "tier1_results.csv", index=False)

# Summary bar chart: XGBoost ablation 
# Ablation bar chart  
xgb_res = results_df[results_df["Model"] == "XGBoost"].set_index("Feature Set")
xgb_res = xgb_res.reindex(["T", "T+O+W", "T+O+W+S"])

metrics_to_plot = ["Balanced Accuracy", "F1-Score", "AUC-ROC"]
PALETTE = ["#4472A8", "#C0504D", "#9BBB59"]

x      = np.arange(len(xgb_res))
n_met  = len(metrics_to_plot)
width  = 0.22
offsets = np.linspace(-(n_met - 1) / 2 * width, (n_met - 1) / 2 * width, n_met)

fig, ax = plt.subplots(figsize=(6.5, 4.0))

for col, offset, colour in zip(metrics_to_plot, offsets, PALETTE):
    bars = ax.bar(
        x + offset,
        xgb_res[col],
        width=width,
        color=colour,
        edgecolor="white",
        linewidth=0.5,
        label=col,
    )
    # Value annotations above each bar
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.008,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=7.5,
            color="#333333",
        )

ax.set_xticks(x)
ax.set_xticklabels(["T", "T+O+W", "T+O+W+S"], fontsize=10)
ax.set_xlabel("Feature set", labelpad=6)
ax.set_ylabel("Score", labelpad=6)
ax.set_ylim(0, 1.05)
ax.set_title(
    "XGBoost — ablation study across feature sets (test set)",
    pad=10,
    fontweight="bold",
)
ax.legend(
    frameon=True,
    framealpha=0.9,
    edgecolor="#cccccc",
    loc="lower right",
)
fig.tight_layout()

fname = RESULTS_DIR / "xgb_ablation_barplot.png"
fig.savefig(fname, dpi=300)
plt.close()
print(f"Ablation plot saved: {fname}")

print(f"\nResults saved to {RESULTS_DIR}")


Feature set: T  (12 features requested)
  Available : 12 features

  ── Logistic Regression ──
  [Logistic Regression | T]  BA=0.5741  F1=0.5776  AUC=0.6750  θ=0.325
               precision    recall  f1-score   support

Not Disrupted       0.85      0.21      0.34      7537
    Disrupted       0.42      0.94      0.58      4544

     accuracy                           0.48     12081
    macro avg       0.63      0.57      0.46     12081
 weighted avg       0.69      0.48      0.43     12081


  ── Random Forest ──
  [Random Forest | T]  BA=0.7194  F1=0.6621  AUC=0.7980  θ=0.379
               precision    recall  f1-score   support

Not Disrupted       0.82      0.69      0.75      7537
    Disrupted       0.59      0.75      0.66      4544

     accuracy                           0.71     12081
    macro avg       0.71      0.72      0.70     12081
 weighted avg       0.73      0.71      0.72     12081


  ── XGBoost ──
  [XGBoost | T]  BA=0.7148  F1=0.6599  AUC=0.7988  θ=0.455
   